# Exploratory Topological Data Analysis

In [1]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path

from gtda.homology import VietorisRipsPersistence
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

In [2]:
# Configuration
DATA_DIR = Path("../../data/raw/fetched")

OHLCV_DIR = DATA_DIR / "asset_ohlcv_5m"
ASSET_DAILY_PATH = DATA_DIR / "asset_daily/asset_daily.parquet"
ASSETS_PATH = DATA_DIR / "assets.parquet"

WINDOW_SIZE_DAYS = 30
MIN_ASSET_COVERAGE = 0.95
MIN_PERSISTENCE = 1.0
MAX_EDGE_LENGTH = 2.0
MIN_CLUSTER_SIZE = 2

DATE_COLUMN = "date"
TIMESTAMP_COLUMN = "timestamp"
CLOSE_COLUMN = "close"
ASSET_ID_COLUMN = "asset_id"
OHLCV_SYMBOL_COLUMN = "market_symbol"
BASE_SYMBOL_COLUMN = "base_symbol"

In [3]:
# Load Data
def read_asset_ohlcv_5m(base_path=OHLCV_DIR) -> pd.DataFrame:
    base = Path(base_path)
    files = sorted(base.glob("ohlcv_*.parquet"))

    if not files:
        raise FileNotFoundError(f"No parquet files found in {base}")

    df = pd.concat((pd.read_parquet(file) for file in files), ignore_index=True)
    return df


ohlcv_df = read_asset_ohlcv_5m()
asset_daily_df = pd.read_parquet(ASSET_DAILY_PATH)
assets_df = pd.read_parquet(ASSETS_PATH)

print("ohlcv_df:", ohlcv_df.shape)
print("asset_daily_df:", asset_daily_df.shape)
print("assets_df:", assets_df.shape)

ohlcv_df: (1038360, 9)
asset_daily_df: (4013, 6)
assets_df: (45, 8)


In [4]:
# Prepare Asset Master Data
assets_df = assets_df.copy()

required_asset_cols = [ASSET_ID_COLUMN, "symbol", "name"]
missing_asset_cols = [col for col in required_asset_cols if col not in assets_df.columns]
if missing_asset_cols:
    raise KeyError(f"Missing required columns in assets.parquet: {missing_asset_cols}")

assets_df["symbol"] = assets_df["symbol"].astype(str).str.upper().str.strip()
assets_df["name"] = assets_df["name"].astype(str).str.strip()

asset_lookup_df = assets_df[[ASSET_ID_COLUMN, "symbol", "name"]].drop_duplicates()

asset_lookup_df.head()

,asset_id,symbol,name
0,1,BTC,Bitcoin
1,2,ETH,Ethereum
2,3,USDT,Tether
3,4,BNB,BNB
4,5,XRP,XRP


In [5]:
# Check lenghts
print(assets_df["symbol"].nunique())
print(asset_daily_df["asset_id"].nunique())
print(ohlcv_df["market_symbol"].nunique())

45
45
41


In [6]:
# Check Asset Matching Across Tables
asset_master_symbols = set(assets_df["symbol"].dropna().astype(str).str.upper().str.strip())
asset_daily_ids = set(asset_daily_df[ASSET_ID_COLUMN].dropna().astype(int))
ohlcv_base_symbols = set(
    ohlcv_df[OHLCV_SYMBOL_COLUMN]
    .dropna()
    .astype(str)
    .str.upper()
    .str.strip()
    .str.replace("USDT", "", regex=False)
)

asset_master_ids = set(assets_df[ASSET_ID_COLUMN].dropna().astype(int))

assets_missing_from_daily = (
    assets_df.loc[~assets_df[ASSET_ID_COLUMN].astype(int).isin(asset_daily_ids), [ASSET_ID_COLUMN, "symbol", "name"]]
    .drop_duplicates()
    .sort_values("symbol")
)

daily_missing_from_assets = (
    asset_daily_df.loc[~asset_daily_df[ASSET_ID_COLUMN].astype(int).isin(asset_master_ids), [ASSET_ID_COLUMN]]
    .drop_duplicates()
    .sort_values(ASSET_ID_COLUMN)
)

assets_missing_from_ohlcv = (
    assets_df.loc[~assets_df["symbol"].isin(ohlcv_base_symbols), [ASSET_ID_COLUMN, "symbol", "name"]]
    .drop_duplicates()
    .sort_values("symbol")
)

ohlcv_missing_from_assets = (
    pd.DataFrame({"base_symbol": sorted(ohlcv_base_symbols - asset_master_symbols)})
)

print("Assets in assets.parquet but missing from asset_daily:", len(assets_missing_from_daily))
display(assets_missing_from_daily)

print("Asset IDs in asset_daily but missing from assets.parquet:", len(daily_missing_from_assets))
display(daily_missing_from_assets)

print("Assets in assets.parquet but missing from OHLCV:", len(assets_missing_from_ohlcv))
display(assets_missing_from_ohlcv)

print("OHLCV base symbols missing from assets.parquet:", len(ohlcv_missing_from_assets))
display(ohlcv_missing_from_assets)

Assets in assets.parquet but missing from asset_daily: 0


,asset_id,symbol,name


Asset IDs in asset_daily but missing from assets.parquet: 0


,asset_id


Assets in assets.parquet but missing from OHLCV: 4


,asset_id,symbol,name
16,17,DAI,Dai
9,10,HYPE,Hyperliquid
19,20,PYUSD,PayPal USD
2,3,USDT,Tether


OHLCV base symbols missing from assets.parquet: 0


,base_symbol


In [7]:
# Prepare OHLCV Data And Map To Asset ID
ohlcv_df = ohlcv_df.copy()

ohlcv_df[TIMESTAMP_COLUMN] = pd.to_datetime(ohlcv_df[TIMESTAMP_COLUMN], unit="ms", utc=True, errors="coerce")
ohlcv_df = ohlcv_df.dropna(subset=[TIMESTAMP_COLUMN, OHLCV_SYMBOL_COLUMN, CLOSE_COLUMN])

ohlcv_df[OHLCV_SYMBOL_COLUMN] = ohlcv_df[OHLCV_SYMBOL_COLUMN].astype(str).str.upper().str.strip()
ohlcv_df[DATE_COLUMN] = ohlcv_df[TIMESTAMP_COLUMN].dt.normalize()

ohlcv_df[BASE_SYMBOL_COLUMN] = ohlcv_df[OHLCV_SYMBOL_COLUMN].str.replace("USDT", "", regex=False)

ohlcv_df = ohlcv_df.merge(
    asset_lookup_df[[ASSET_ID_COLUMN, "symbol"]].rename(columns={"symbol": BASE_SYMBOL_COLUMN}),
    on=BASE_SYMBOL_COLUMN,
    how="left",
)

unmatched_symbols = (
    ohlcv_df.loc[ohlcv_df[ASSET_ID_COLUMN].isna(), [OHLCV_SYMBOL_COLUMN, BASE_SYMBOL_COLUMN]]
    .drop_duplicates()
    .sort_values([OHLCV_SYMBOL_COLUMN])
)

print("Unmatched OHLCV symbols:", len(unmatched_symbols))
display(unmatched_symbols.head(20))

ohlcv_df = ohlcv_df.dropna(subset=[ASSET_ID_COLUMN]).copy()
ohlcv_df[ASSET_ID_COLUMN] = ohlcv_df[ASSET_ID_COLUMN].astype("Int64")

ohlcv_df[[TIMESTAMP_COLUMN, DATE_COLUMN, OHLCV_SYMBOL_COLUMN, BASE_SYMBOL_COLUMN, ASSET_ID_COLUMN, CLOSE_COLUMN]].head()

Unmatched OHLCV symbols: 0


,market_symbol,base_symbol


,timestamp,date,market_symbol,base_symbol,asset_id,close
0,2026-01-01 00:00:00+00:00,2026-01-01 00:00:00+00:00,AAVEUSDT,AAVE,34,146.22
1,2026-01-01 00:05:00+00:00,2026-01-01 00:00:00+00:00,AAVEUSDT,AAVE,34,146.39
2,2026-01-01 00:10:00+00:00,2026-01-01 00:00:00+00:00,AAVEUSDT,AAVE,34,146.34
3,2026-01-01 00:15:00+00:00,2026-01-01 00:00:00+00:00,AAVEUSDT,AAVE,34,146.38
4,2026-01-01 00:20:00+00:00,2026-01-01 00:00:00+00:00,AAVEUSDT,AAVE,34,146.54


In [8]:
# Prepare Universe Data
asset_daily_df = asset_daily_df.copy()

if DATE_COLUMN not in asset_daily_df.columns:
    raise KeyError(f"'{DATE_COLUMN}' not found in asset_daily.parquet")

if ASSET_ID_COLUMN not in asset_daily_df.columns:
    raise KeyError(f"'{ASSET_ID_COLUMN}' not found in asset_daily.parquet")

asset_daily_df[DATE_COLUMN] = pd.to_datetime(asset_daily_df[DATE_COLUMN], utc=True, errors="coerce").dt.normalize()
asset_daily_df = asset_daily_df.dropna(subset=[DATE_COLUMN, ASSET_ID_COLUMN]).copy()
asset_daily_df[ASSET_ID_COLUMN] = asset_daily_df[ASSET_ID_COLUMN].astype("Int64")

universe_df = asset_daily_df[[DATE_COLUMN, ASSET_ID_COLUMN]].drop_duplicates().copy()

print("universe_df:", universe_df.shape)
universe_df.head()

universe_df: (4013, 2)


,date,asset_id
0,2026-01-01 00:00:00+00:00,1
1,2026-01-01 00:00:00+00:00,2
2,2026-01-01 00:00:00+00:00,3
3,2026-01-01 00:00:00+00:00,4
4,2026-01-01 00:00:00+00:00,5


In [9]:
# Check Universe Coverage In OHLCV
universe_asset_ids = set(universe_df[ASSET_ID_COLUMN].dropna().astype(int).unique())
ohlcv_asset_ids = set(ohlcv_df[ASSET_ID_COLUMN].dropna().astype(int).unique())

print("Universe asset_ids:", len(universe_asset_ids))
print("OHLCV asset_ids:", len(ohlcv_asset_ids))
print("Intersection:", len(universe_asset_ids & ohlcv_asset_ids))
print("Universe asset_ids missing in OHLCV:", len(universe_asset_ids - ohlcv_asset_ids))

Universe asset_ids: 45
OHLCV asset_ids: 41
Intersection: 41
Universe asset_ids missing in OHLCV: 4


In [10]:
# Helper Functions
def correlation_distance(returns_window: pd.DataFrame) -> np.ndarray:
    corr = np.corrcoef(returns_window.to_numpy().T)
    corr = np.nan_to_num(corr, nan=0.0)
    corr = np.clip(corr, -1.0, 1.0)
    dist = np.sqrt(np.maximum(0.0, 2.0 * (1.0 - corr)))
    np.fill_diagonal(dist, 0.0)
    return dist


def get_universe_asset_ids_for_date(universe_df: pd.DataFrame, end_date: pd.Timestamp) -> list[int]:
    asset_ids = (
        universe_df.loc[universe_df[DATE_COLUMN] == end_date, ASSET_ID_COLUMN]
        .dropna()
        .astype(int)
        .sort_values()
        .unique()
        .tolist()
    )
    return asset_ids


def build_window_input(
    ohlcv_df: pd.DataFrame,
    universe_df: pd.DataFrame,
    end_date: pd.Timestamp,
    window_size_days: int = WINDOW_SIZE_DAYS,
    min_asset_coverage: float = MIN_ASSET_COVERAGE,
) -> dict | None:
    end_date = pd.to_datetime(end_date, utc=True).normalize()
    start_date = end_date - pd.Timedelta(days=window_size_days - 1)

    eligible_asset_ids = get_universe_asset_ids_for_date(universe_df, end_date)
    if len(eligible_asset_ids) < 2:
        return None

    window_data = ohlcv_df.loc[
        (ohlcv_df[DATE_COLUMN] >= start_date) &
        (ohlcv_df[DATE_COLUMN] <= end_date) &
        (ohlcv_df[ASSET_ID_COLUMN].isin(eligible_asset_ids)),
        [TIMESTAMP_COLUMN, ASSET_ID_COLUMN, CLOSE_COLUMN]
    ].copy()

    if window_data.empty:
        return None

    price_matrix = (
        window_data
        .pivot_table(
            index=TIMESTAMP_COLUMN,
            columns=ASSET_ID_COLUMN,
            values=CLOSE_COLUMN,
            aggfunc="last",
        )
        .sort_index()
    )

    if price_matrix.empty:
        return None

    min_non_null = int(np.ceil(len(price_matrix) * min_asset_coverage))
    price_matrix = price_matrix.dropna(axis=1, thresh=min_non_null)
    price_matrix = price_matrix.dropna(axis=0, how="any")

    if price_matrix.shape[0] < 3 or price_matrix.shape[1] < 2:
        return None

    returns = price_matrix.pct_change().dropna(how="any")

    if returns.shape[0] < 2 or returns.shape[1] < 2:
        return None

    dist = correlation_distance(returns)

    return {
        "start_date": start_date,
        "end_date": end_date,
        "asset_ids": [int(x) for x in returns.columns.tolist()],
        "n_assets": returns.shape[1],
        "n_timestamps": returns.shape[0],
        "price_matrix": price_matrix,
        "returns": returns,
        "distance_matrix": dist,
    }


def extract_persistent_h0_features(diagram: np.ndarray, min_persistence: float = MIN_PERSISTENCE) -> list[dict]:
    if diagram.size == 0:
        return []

    diagram_df = pd.DataFrame(diagram, columns=["birth", "death", "dimension"])

    h0 = diagram_df.loc[diagram_df["dimension"] == 0].copy()

    h0["persistence"] = h0["death"] - h0["birth"]

    h0 = h0.loc[h0["persistence"] > min_persistence].copy()
    h0 = h0.sort_values("persistence", ascending=False)

    return h0[["birth", "death", "persistence"]].to_dict("records")


def get_clusters_at_scale(
    dist_matrix: np.ndarray,
    asset_ids: list[int],
    scale: float,
    min_cluster_size: int = MIN_CLUSTER_SIZE,
) -> list[list[int]]:
    if len(asset_ids) < 2:
        return []

    condensed = squareform(dist_matrix, checks=False)
    z = linkage(condensed, method="single")
    labels = fcluster(z, t=scale, criterion="distance")

    groups = {}
    for idx, label in enumerate(labels):
        groups.setdefault(label, []).append(int(asset_ids[idx]))

    clusters = [members for members in groups.values() if len(members) >= min_cluster_size]
    clusters = sorted(clusters, key=lambda x: (-len(x), x))
    return clusters

In [11]:
# Available Window End Dates
available_end_dates = sorted(universe_df[DATE_COLUMN].dropna().unique())

len(available_end_dates), available_end_dates[:3], available_end_dates[-3:]

(90,
 [Timestamp('2026-01-01 00:00:00+0000', tz='UTC'),
  Timestamp('2026-01-02 00:00:00+0000', tz='UTC'),
  Timestamp('2026-01-03 00:00:00+0000', tz='UTC')],
 [Timestamp('2026-03-29 00:00:00+0000', tz='UTC'),
  Timestamp('2026-03-30 00:00:00+0000', tz='UTC'),
  Timestamp('2026-03-31 00:00:00+0000', tz='UTC')])

In [12]:
# Build All Window Inputs
window_inputs = []

for end_date in available_end_dates:
    window_result = build_window_input(
        ohlcv_df=ohlcv_df,
        universe_df=universe_df,
        end_date=end_date,
        window_size_days=WINDOW_SIZE_DAYS,
        min_asset_coverage=MIN_ASSET_COVERAGE,
    )

    if window_result is not None:
        window_inputs.append(window_result)

len(window_inputs)

90

In [13]:
# Window Summary
window_summary = pd.DataFrame(
    {
        "start_date": [w["start_date"] for w in window_inputs],
        "end_date": [w["end_date"] for w in window_inputs],
        "n_assets": [w["n_assets"] for w in window_inputs],
        "n_timestamps": [w["n_timestamps"] for w in window_inputs],
    }
)

window_summary.head()

,start_date,end_date,n_assets,n_timestamps
0,2025-12-03 00:00:00+00:00,2026-01-01 00:00:00+00:00,40,287
1,2025-12-04 00:00:00+00:00,2026-01-02 00:00:00+00:00,40,575
2,2025-12-05 00:00:00+00:00,2026-01-03 00:00:00+00:00,40,863
3,2025-12-06 00:00:00+00:00,2026-01-04 00:00:00+00:00,40,1151
4,2025-12-07 00:00:00+00:00,2026-01-05 00:00:00+00:00,40,1439


In [14]:
# Inspect One Window
example_idx = 0

window_inputs[example_idx]["start_date"], window_inputs[example_idx]["end_date"], window_inputs[example_idx]["n_assets"]

(Timestamp('2025-12-03 00:00:00+0000', tz='UTC'),
 Timestamp('2026-01-01 00:00:00+0000', tz='UTC'),
 40)

In [15]:
# Inspect One Window Asset IDs
window_inputs[example_idx]["asset_ids"][:20]

[1, 2, 4, 5, 6, 7, 8, 9, 11, 12, 13, 14, 15, 16, 18, 19, 21, 22, 23, 24]

In [16]:
# Run Vietoris-Rips Per Window
vr = VietorisRipsPersistence(
    metric="precomputed",
    homology_dimensions=[0],
    max_edge_length=MAX_EDGE_LENGTH,
    n_jobs=-1,
)

persistent_results = []

for window in window_inputs:
    dist_matrix = window["distance_matrix"]
    diagram = vr.fit_transform(dist_matrix[None, :, :])[0]

    features = extract_persistent_h0_features(
        diagram=diagram,
        min_persistence=MIN_PERSISTENCE,
    )

    persistent_results.append(
        {
            "start_date": window["start_date"],
            "end_date": window["end_date"],
            "asset_ids": window["asset_ids"],
            "n_assets": window["n_assets"],
            "n_timestamps": window["n_timestamps"],
            "distance_matrix": dist_matrix,
            "diagram": diagram,
            "persistent_features": features,
        }
    )

len(persistent_results)

90

In [17]:
# Persistent Feature Summary
persistent_summary = pd.DataFrame(
    {
        "start_date": [r["start_date"] for r in persistent_results],
        "end_date": [r["end_date"] for r in persistent_results],
        "n_assets": [r["n_assets"] for r in persistent_results],
        "n_timestamps": [r["n_timestamps"] for r in persistent_results],
        "n_persistent_h0_features": [len(r["persistent_features"]) for r in persistent_results],
        "max_persistence": [
            max((f["persistence"] for f in r["persistent_features"]), default=np.nan)
            for r in persistent_results
        ],
    }
)

persistent_summary.head()

,start_date,end_date,n_assets,n_timestamps,n_persistent_h0_features,max_persistence
0,2025-12-03 00:00:00+00:00,2026-01-01 00:00:00+00:00,40,287,18,1.348858
1,2025-12-04 00:00:00+00:00,2026-01-02 00:00:00+00:00,40,575,13,1.378680
2,2025-12-05 00:00:00+00:00,2026-01-03 00:00:00+00:00,40,863,14,1.378440
3,2025-12-06 00:00:00+00:00,2026-01-04 00:00:00+00:00,40,1151,14,1.392747
4,2025-12-07 00:00:00+00:00,2026-01-05 00:00:00+00:00,40,1439,13,1.399886


In [18]:
# Map Persistent Features To Asset Groups
CLUSTER_SCALE_MODE = "midpoint"
# Options: "birth", "death", "midpoint"

cluster_results = []

for result in persistent_results:
    dist_matrix = result["distance_matrix"]
    asset_ids = result["asset_ids"]

    clusters_for_window = []

    for feature in result["persistent_features"]:
        if CLUSTER_SCALE_MODE == "birth":
            scale = feature["birth"]
        elif CLUSTER_SCALE_MODE == "death":
            scale = feature["death"]
        elif CLUSTER_SCALE_MODE == "midpoint":
            scale = feature["birth"] + 0.5 * feature["persistence"]
        else:
            raise ValueError("CLUSTER_SCALE_MODE must be 'birth', 'death', or 'midpoint'")

        groups = get_clusters_at_scale(
            dist_matrix=dist_matrix,
            asset_ids=asset_ids,
            scale=scale,
            min_cluster_size=MIN_CLUSTER_SIZE,
        )

        clusters_for_window.append(
            {
                "birth": feature["birth"],
                "death": feature["death"],
                "persistence": feature["persistence"],
                "scale_mode": CLUSTER_SCALE_MODE,
                "scale_used": scale,
                "groups": groups,
            }
        )

    cluster_results.append(
        {
            "start_date": result["start_date"],
            "end_date": result["end_date"],
            "n_assets": result["n_assets"],
            "n_timestamps": result["n_timestamps"],
            "clusters": clusters_for_window,
        }
    )

len(cluster_results)

90

In [19]:
# Flatten Cluster Output
cluster_rows = []

for result in cluster_results:
    for feature_idx, feature in enumerate(result["clusters"], start=1):
        for group_idx, group in enumerate(feature["groups"], start=1):
            cluster_rows.append(
                {
                    "start_date": result["start_date"],
                    "end_date": result["end_date"],
                    "feature_rank": feature_idx,
                    "birth": feature["birth"],
                    "death": feature["death"],
                    "persistence": feature["persistence"],
                    "scale_mode": feature["scale_mode"],
                    "scale_used": feature["scale_used"],
                    "group_rank": group_idx,
                    "group_size": len(group),
                    "asset_ids": group,
                }
            )

cluster_table = pd.DataFrame(
    cluster_rows,
    columns=[
        "start_date",
        "end_date",
        "feature_rank",
        "birth",
        "death",
        "persistence",
        "scale_mode",
        "scale_used",
        "group_rank",
        "group_size",
        "asset_ids",
    ],
)

print("cluster_table rows:", len(cluster_table))
cluster_table.head()

cluster_table rows: 706


,start_date,end_date,feature_rank,birth,death,persistence,scale_mode,scale_used,group_rank,group_size,asset_ids
0,2025-12-04 00:00:00+00:00,2026-01-02 00:00:00+00:00,1,0.0,1.378680,1.378680,midpoint,0.689340,1,3,"[1, 2, 7]"
1,2025-12-04 00:00:00+00:00,2026-01-02 00:00:00+00:00,2,0.0,1.342136,1.342136,midpoint,0.671068,1,3,"[1, 2, 7]"
2,2025-12-04 00:00:00+00:00,2026-01-02 00:00:00+00:00,3,0.0,1.335566,1.335566,midpoint,0.667783,1,3,"[1, 2, 7]"
3,2025-12-04 00:00:00+00:00,2026-01-02 00:00:00+00:00,4,0.0,1.306683,1.306683,midpoint,0.653341,1,3,"[1, 2, 7]"
4,2025-12-04 00:00:00+00:00,2026-01-02 00:00:00+00:00,5,0.0,1.295093,1.295093,midpoint,0.647546,1,3,"[1, 2, 7]"


In [20]:
# Check Persistent Feature Counts
persistent_summary[["end_date", "n_assets", "n_persistent_h0_features", "max_persistence"]].head(20)

,end_date,n_assets,n_persistent_h0_features,max_persistence
0,2026-01-01 00:00:00+00:00,40,18,1.348858
1,2026-01-02 00:00:00+00:00,40,13,1.378680
2,2026-01-03 00:00:00+00:00,40,14,1.378440
3,2026-01-04 00:00:00+00:00,40,14,1.392747
4,2026-01-05 00:00:00+00:00,40,13,1.399886
5,2026-01-06 00:00:00+00:00,40,11,1.407062
6,2026-01-07 00:00:00+00:00,40,10,1.406508
7,2026-01-08 00:00:00+00:00,40,10,1.407044
8,2026-01-09 00:00:00+00:00,40,10,1.407392
9,2026-01-10 00:00:00+00:00,40,10,1.408079


In [21]:
# Add Readable Asset Labels
asset_name_lookup = asset_lookup_df.rename(columns={"symbol": "asset_symbol", "name": "asset_name"})

def enrich_asset_ids(asset_ids: list[int], asset_lookup: pd.DataFrame) -> list[dict]:
    lookup = asset_lookup.set_index(ASSET_ID_COLUMN)[["asset_symbol", "asset_name"]].to_dict("index")
    enriched = []

    for asset_id in asset_ids:
        asset_id = int(asset_id)
        info = lookup.get(asset_id, {})
        enriched.append(
            {
                "asset_id": asset_id,
                "symbol": info.get("asset_symbol"),
                "name": info.get("asset_name"),
            }
        )

    return enriched


if cluster_table.empty:
    print("No clusters found. Try CLUSTER_SCALE_MODE = 'midpoint' or 'death', or lower MIN_CLUSTER_SIZE.")
else:
    cluster_table["assets"] = cluster_table["asset_ids"].apply(
        lambda ids: enrich_asset_ids(ids, asset_name_lookup)
    )

cluster_table.head()

,start_date,end_date,feature_rank,birth,death,persistence,scale_mode,scale_used,group_rank,group_size,asset_ids,assets
0,2025-12-04 00:00:00+00:00,2026-01-02 00:00:00+00:00,1,0.0,1.378680,1.378680,midpoint,0.689340,1,3,"[1, 2, 7]","[{'asset_id': 1, 'symbol': 'BTC', 'name': 'Bit..."
1,2025-12-04 00:00:00+00:00,2026-01-02 00:00:00+00:00,2,0.0,1.342136,1.342136,midpoint,0.671068,1,3,"[1, 2, 7]","[{'asset_id': 1, 'symbol': 'BTC', 'name': 'Bit..."
2,2025-12-04 00:00:00+00:00,2026-01-02 00:00:00+00:00,3,0.0,1.335566,1.335566,midpoint,0.667783,1,3,"[1, 2, 7]","[{'asset_id': 1, 'symbol': 'BTC', 'name': 'Bit..."
3,2025-12-04 00:00:00+00:00,2026-01-02 00:00:00+00:00,4,0.0,1.306683,1.306683,midpoint,0.653341,1,3,"[1, 2, 7]","[{'asset_id': 1, 'symbol': 'BTC', 'name': 'Bit..."
4,2025-12-04 00:00:00+00:00,2026-01-02 00:00:00+00:00,5,0.0,1.295093,1.295093,midpoint,0.647546,1,3,"[1, 2, 7]","[{'asset_id': 1, 'symbol': 'BTC', 'name': 'Bit..."


In [22]:
# Largest Clusters Per Window
largest_clusters = (
    cluster_table
    .sort_values(["end_date", "group_size", "persistence"], ascending=[True, False, False])
    .groupby("end_date", as_index=False)
    .first()
)

largest_clusters[["end_date", "group_size", "persistence", "assets"]].head()

,end_date,group_size,persistence,assets
0,2026-01-02 00:00:00+00:00,3,1.378680,"[{'asset_id': 1, 'symbol': 'BTC', 'name': 'Bit..."
1,2026-01-03 00:00:00+00:00,3,1.378440,"[{'asset_id': 1, 'symbol': 'BTC', 'name': 'Bit..."
2,2026-01-04 00:00:00+00:00,3,1.392747,"[{'asset_id': 1, 'symbol': 'BTC', 'name': 'Bit..."
3,2026-01-05 00:00:00+00:00,3,1.399886,"[{'asset_id': 1, 'symbol': 'BTC', 'name': 'Bit..."
4,2026-01-06 00:00:00+00:00,12,1.407062,"[{'asset_id': 1, 'symbol': 'BTC', 'name': 'Bit..."


In [23]:
# Final Results Overview

print("=== Persistent Feature Summary ===")
display(
    persistent_summary[
        ["end_date", "n_assets", "n_persistent_h0_features", "max_persistence"]
    ].tail(20)
)

print("\n=== Largest Clusters Per Window ===")
display(
    largest_clusters[
        ["end_date", "group_size", "persistence", "assets"]
    ].tail(10)
)

print("\n=== Max Cluster Size Over Time ===")
display(
    cluster_table.groupby("end_date")["group_size"].max().tail(20)
)

=== Persistent Feature Summary ===


,end_date,n_assets,n_persistent_h0_features,max_persistence
70,2026-03-12 00:00:00+00:00,39,5,1.400692
71,2026-03-13 00:00:00+00:00,39,5,1.398825
72,2026-03-14 00:00:00+00:00,39,7,1.398179
73,2026-03-15 00:00:00+00:00,39,7,1.398662
74,2026-03-16 00:00:00+00:00,39,7,1.399292
75,2026-03-17 00:00:00+00:00,39,7,1.398595
76,2026-03-18 00:00:00+00:00,39,8,1.397124
77,2026-03-19 00:00:00+00:00,39,7,1.394646
78,2026-03-20 00:00:00+00:00,39,7,1.394864
79,2026-03-21 00:00:00+00:00,39,7,1.394714



=== Largest Clusters Per Window ===


,end_date,group_size,persistence,assets
79,2026-03-22 00:00:00+00:00,17,1.394860,"[{'asset_id': 1, 'symbol': 'BTC', 'name': 'Bit..."
80,2026-03-23 00:00:00+00:00,18,1.395902,"[{'asset_id': 1, 'symbol': 'BTC', 'name': 'Bit..."
81,2026-03-24 00:00:00+00:00,18,1.396061,"[{'asset_id': 1, 'symbol': 'BTC', 'name': 'Bit..."
82,2026-03-25 00:00:00+00:00,18,1.410746,"[{'asset_id': 1, 'symbol': 'BTC', 'name': 'Bit..."
83,2026-03-26 00:00:00+00:00,17,1.407724,"[{'asset_id': 1, 'symbol': 'BTC', 'name': 'Bit..."
84,2026-03-27 00:00:00+00:00,17,1.406761,"[{'asset_id': 1, 'symbol': 'BTC', 'name': 'Bit..."
85,2026-03-28 00:00:00+00:00,17,1.408260,"[{'asset_id': 1, 'symbol': 'BTC', 'name': 'Bit..."
86,2026-03-29 00:00:00+00:00,17,1.407090,"[{'asset_id': 1, 'symbol': 'BTC', 'name': 'Bit..."
87,2026-03-30 00:00:00+00:00,17,1.407259,"[{'asset_id': 1, 'symbol': 'BTC', 'name': 'Bit..."
88,2026-03-31 00:00:00+00:00,16,1.407053,"[{'asset_id': 1, 'symbol': 'BTC', 'name': 'Bit..."



=== Max Cluster Size Over Time ===


end_date
2026-03-12 00:00:00+00:00    20
2026-03-13 00:00:00+00:00    20
2026-03-14 00:00:00+00:00    19
2026-03-15 00:00:00+00:00    18
2026-03-16 00:00:00+00:00    18
2026-03-17 00:00:00+00:00    18
2026-03-18 00:00:00+00:00    18
2026-03-19 00:00:00+00:00    18
2026-03-20 00:00:00+00:00    18
2026-03-21 00:00:00+00:00    18
2026-03-22 00:00:00+00:00    17
2026-03-23 00:00:00+00:00    18
2026-03-24 00:00:00+00:00    18
2026-03-25 00:00:00+00:00    18
2026-03-26 00:00:00+00:00    17
2026-03-27 00:00:00+00:00    17
2026-03-28 00:00:00+00:00    17
2026-03-29 00:00:00+00:00    17
2026-03-30 00:00:00+00:00    17
2026-03-31 00:00:00+00:00    16
Name: group_size, dtype: int64